In [145]:
import pandas as pd

df = pd.read_csv('issues.csv')
import json

columns_to_keep = [
    "url", "comments_url", "number", "title", "user", "labels",
    "state", "assignee", "comments", "created_at", "updated_at",
    "closed_at", "timeline_url"
]
df = df[columns_to_keep]
def transform_url(api_url):
    try:
        return api_url.replace("https://api.github.com/repos/", "https://github.com/").replace("/issues/", "/issues/")
    except AttributeError:
        return api_url
df["url"] = df["url"].apply(transform_url)

def extract_json_field(json_data, key):
    try:
        if isinstance(json_data, str):
            json_data = json.loads(json_data.replace("'", "\""))
        if isinstance(json_data, list):
            return [item.get(key, None) for item in json_data]
        return json_data.get(key, None)
    except Exception as e:
        return None

import ast

df['user'] = df['user'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else {})
df['user'] = df['user'].apply(lambda x: x.get("login") if isinstance(x, dict) else None)

if 'assignee' in df.columns:
    df['assignee'] = df['assignee'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else {})
    df['assignee'] = df['assignee'].apply(lambda x: x.get("login") if isinstance(x, dict) else None)

df['labels'] = df['labels'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])

df['labels_name'] = df['labels'].apply(lambda x: [d['name'] for d in x] if isinstance(x, list) else [])
df['labels_description'] = df['labels'].apply(lambda x: [d['description'] for d in x] if isinstance(x, list) else [])


df.drop('labels', axis=1, inplace=True)
df.drop('timeline_url', axis=1, inplace=True)
df.drop('comments_url', axis=1, inplace=True)

In [146]:
import re
def process_tags(heading):
    tags = re.findall(r'\[.*?\]', heading)
    cleaned_tags = [re.sub(r'[^A-Za-z]', '', tag).upper() for tag in tags]
    cleaned_tags = [tag for tag in cleaned_tags if len(tag) <= 13]
    return cleaned_tags
df['processed_tags'] = df['title'].apply(process_tags)

,url,number,title,user,state,assignee,comments,created_at,updated_at,closed_at,labels_name,labels_description,processed_tags
0,https://github.com/hyprbots/engineering-backlo...,2569,[database] add mongodb triggers,hansal-hyprbots,open,hansal-hyprbots,0,2025-01-13T09:33:25Z,2025-01-13T09:33:25Z,NaN,[],[],[DATABASE]
1,https://github.com/hyprbots/engineering-backlo...,2568,[gl-recommendation] gl key - gl number integra...,hansal-hyprbots,open,hansal-hyprbots,0,2025-01-13T09:32:57Z,2025-01-13T09:32:57Z,NaN,[],[],[]
2,https://github.com/hyprbots/engineering-backlo...,2567,[work-item] new invoice edit api,hansal-hyprbots,open,hansal-hyprbots,0,2025-01-13T09:32:51Z,2025-01-13T09:32:53Z,NaN,[],[],[WORKITEM]
3,https://github.com/hyprbots/engineering-backlo...,2566,"Change workflow response as per new response, ...",mohit-raj355,open,mohit-raj355,0,2025-01-13T09:31:17Z,2025-01-13T09:31:17Z,NaN,[],[],[]
4,https://github.com/hyprbots/engineering-backlo...,2565,[BUGS][PFMT][PROD][Accruals -> After updating ...,satyam-hyperbots,open,mohit-raj355,0,2025-01-13T09:27:16Z,2025-01-13T09:27:16Z,NaN,[],[],"[BUGS, PFMT, PROD]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2543,https://github.com/hyprbots/engineering-backlo...,5,Tally Connector v1 Implementation,ram-hyprbots,open,ram-hyprbots,0,2023-10-16T08:58:09Z,2023-10-16T15:42:02Z,NaN,[],[],[]
2544,https://github.com/hyprbots/engineering-backlo...,4,Tally Connector v1 Design,ram-hyprbots,open,ram-hyprbots,0,2023-10-16T08:58:00Z,2023-10-16T15:42:32Z,NaN,[],[],[]
2545,https://github.com/hyprbots/engineering-backlo...,3,Book Invoices Task v1 Implementation,ram-hyprbots,open,ram-hyprbots,0,2023-10-16T08:57:49Z,2023-10-16T15:42:20Z,NaN,[],[],[]
2546,https://github.com/hyprbots/engineering-backlo...,2,Email Verification Implementation,ram-hyprbots,open,PriyankHyprbots,0,2023-10-16T08:57:40Z,2023-10-16T08:57:40Z,NaN,[],[],[]


In [147]:
df.head(20)

,url,number,title,user,state,assignee,comments,created_at,updated_at,closed_at,labels_name,labels_description,processed_tags
0,https://github.com/hyprbots/engineering-backlo...,2569,[database] add mongodb triggers,hansal-hyprbots,open,hansal-hyprbots,0,2025-01-13T09:33:25Z,2025-01-13T09:33:25Z,NaN,[],[],[DATABASE]
1,https://github.com/hyprbots/engineering-backlo...,2568,[gl-recommendation] gl key - gl number integra...,hansal-hyprbots,open,hansal-hyprbots,0,2025-01-13T09:32:57Z,2025-01-13T09:32:57Z,NaN,[],[],[]
2,https://github.com/hyprbots/engineering-backlo...,2567,[work-item] new invoice edit api,hansal-hyprbots,open,hansal-hyprbots,0,2025-01-13T09:32:51Z,2025-01-13T09:32:53Z,NaN,[],[],[WORKITEM]
3,https://github.com/hyprbots/engineering-backlo...,2566,"Change workflow response as per new response, ...",mohit-raj355,open,mohit-raj355,0,2025-01-13T09:31:17Z,2025-01-13T09:31:17Z,NaN,[],[],[]
4,https://github.com/hyprbots/engineering-backlo...,2565,[BUGS][PFMT][PROD][Accruals -> After updating ...,satyam-hyperbots,open,mohit-raj355,0,2025-01-13T09:27:16Z,2025-01-13T09:27:16Z,NaN,[],[],"[BUGS, PFMT, PROD]"
5,https://github.com/hyprbots/engineering-backlo...,2564,[Bugs][Accrual][Created By filter page says 'N...,adi-anup,open,None,0,2025-01-13T07:45:45Z,2025-01-13T07:45:59Z,NaN,[Accruals],[Label for Accrual Task],"[BUGS, ACCRUAL]"
6,https://github.com/hyprbots/engineering-backlo...,2563,[BUGS][PFMT][PROD][PR/PO -> Export all data ->...,satyam-hyperbots,open,parshva-hyprbots,0,2025-01-13T07:27:49Z,2025-01-13T07:27:49Z,NaN,[],[],"[BUGS, PFMT, PROD]"
7,https://github.com/hyprbots/engineering-backlo...,2562,[Bugs][Accruals][Manual expense got booked as ...,adi-anup,open,None,0,2025-01-13T07:24:55Z,2025-01-13T07:25:44Z,NaN,[Accruals],[Label for Accrual Task],"[BUGS, ACCRUALS]"
8,https://github.com/hyprbots/engineering-backlo...,2561,"[BUGS][PFMT][PROD][PR/PO -> ""Pending On"" tab -...",satyam-hyperbots,open,parshva-hyprbots,0,2025-01-13T07:24:12Z,2025-01-13T07:24:12Z,NaN,[],[],"[BUGS, PFMT, PROD]"
9,https://github.com/hyprbots/engineering-backlo...,2560,[Bugs] [Invoice] [GL account has some undefine...,Ammaar1904,open,None,0,2025-01-13T06:39:17Z,2025-01-13T06:39:17Z,NaN,[],[],"[BUGS, INVOICE]"


In [148]:
df.to_csv('issues-processed.csv', index=False)

In [152]:
import pandas as pd
from datetime import datetime
import pytz

# Assuming you already have a DataFrame 'df' with columns 'updated_at', 'created_at', 'closed_at'

def ensure_utc_timezone(dt):
    if pd.notna(dt):
        if dt.tzinfo is None:
            return dt.tz_localize('UTC')
        return dt.tz_convert('UTC')
    return dt

# Convert the timestamp columns to datetime
df['created_at'] = pd.to_datetime(df['created_at'])
df['updated_at'] = pd.to_datetime(df['updated_at'])
df['closed_at'] = pd.to_datetime(df['closed_at'])

# Ensure all datetime columns are time zone-aware (UTC)
df['created_at'] = df['created_at'].apply(ensure_utc_timezone)
df['updated_at'] = df['updated_at'].apply(ensure_utc_timezone)
df['closed_at'] = df['closed_at'].apply(ensure_utc_timezone)

# Current date (timezone-aware)
current_date = datetime.now(pytz.UTC)

# Calculate the age of each bug
df['age'] = (current_date - df['created_at']).dt.days

# Remove rows where the bug is closed
df = df[df['closed_at'].isna()]

# Drop the 'closed_at' column as it's no longer needed
df.drop('closed_at', axis=1, inplace=True)

# Now 'df' contains only open bugs with their age calculated

# Optional: Save the filtered DataFrame to a new CSV file
df.to_csv("open_issues.csv", index=False)
